In [ ]:
import pandas as pd
from pathlib import Path

print(Path.cwd())

columns = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week",
    "native-country", "income",
]

df = pd.read_csv(
    "Adult Data/dataset/adult.data",
    names=columns,
    header=None,
    skipinitialspace=True,
    na_values=["?"],
)

In [9]:
print("Shape:", df.shape)

summary = pd.DataFrame({
    "dtype": df.dtypes,
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2),
    "unique_nonmissing": df.nunique(),
})

display(summary)
display(df.head())

Shape: (32561, 15)


,dtype,missing_count,missing_percent,unique_nonmissing
age,int64,0,0.00,73
workclass,str,1836,5.64,8
fnlwgt,int64,0,0.00,21648
education,str,0,0.00,16
education-num,int64,0,0.00,16
marital-status,str,0,0.00,7
occupation,str,1843,5.66,14
relationship,str,0,0.00,6
race,str,0,0.00,5
sex,str,0,0.00,2


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [10]:
display(summary.loc[summary["missing_count"] > 0])

,dtype,missing_count,missing_percent,unique_nonmissing
workclass,str,1836,5.64,8
occupation,str,1843,5.66,14
native-country,str,583,1.79,41


In [11]:
missing_workclass = df["workclass"].isna()

display(
    pd.crosstab(
        missing_workclass,
        df["income"],
        normalize="index",
    )
)

income,<=50K,>50K
workclass,,
False,0.751017,0.248983
True,0.895969,0.104031


In [12]:
df["income"].value_counts(normalize=True).mul(100)

income
<=50K    75.919044
>50K     24.080956
Name: proportion, dtype: float64

In [ ]:
check = df.assign(
    age_band=pd.cut(
        df["age"],
        bins=[0, 25, 40, 60, float("inf")],
        right=False,
        labels=["<25", "25–39", "40–59", "60+"],
    ),
    workclass_missing=df["workclass"].isna(),
    high_income=df["income"].eq(">50K"),
)

display(
    check.groupby(
        ["age_band", "workclass_missing"],
        observed=True,
    ).agg(
        count=("high_income", "size"),
        high_income_rate=("high_income", "mean"),
    )
)

count  high_income_rate
age_band workclass_missing                         
<25      False               4938          0.011948
         True                 632          0.003165
25–39    False              12381          0.220418
         True                 373          0.080429
40–59    False              11286          0.381535
         True                 307          0.205212
60+      False               2120          0.262264
         True                 524          0.183206

In [14]:
label_map = {"<=50K": 0, ">50K": 1}

y = df["income"].map(label_map)

assert y.notna().all(), "Unexpected or missing income labels"
y = y.astype(int)

X = df.drop(columns=["income"]).copy()

print("Predictors:", X.shape)
print("Target:", y.shape)
print("Positive-class proportion:", y.mean())

Predictors: (32561, 14)
Target: (32561,)
Positive-class proportion: 0.2408095574460244


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

assert not X[numeric_features].isna().any().any(), \
    "Numeric missing values found: inspect before proceeding"

categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(
        strategy="constant",
        fill_value="Unknown",
    )),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

baseline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(C=1.0, max_iter=2000)),
])

baseline

Numeric: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default

In [17]:
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


In [18]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=0,
)

folds = list(cv.split(X, y))

In [19]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate

models = {
    "Dummy": DummyClassifier(strategy="prior"),
    "Logistic regression": baseline,
}

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "accuracy": "accuracy",
}

cv_results = {}
rows = []

for name, model in models.items():
    print(f"Evaluating {name}...")

    result = cross_validate(
        model,
        X,
        y,
        cv=folds,
        scoring=scoring,
        return_train_score=True,
        error_score="raise",
        n_jobs=2,
    )
    cv_results[name] = result

    for metric in scoring:
        rows.append({
            "model": name,
            "metric": metric,
            "train_mean": result[f"train_{metric}"].mean(),
            "validation_mean": result[f"test_{metric}"].mean(),
            "validation_std": result[f"test_{metric}"].std(),
        })

display(pd.DataFrame(rows).round(4))

Evaluating Dummy...
Evaluating Logistic regression...


,model,metric,train_mean,validation_mean,validation_std
0,Dummy,roc_auc,0.5000,0.5000,0.0000
1,Dummy,average_precision,0.2408,0.2408,0.0000
2,Dummy,accuracy,0.7592,0.7592,0.0000
3,Logistic regression,roc_auc,0.9089,0.9070,0.0037
4,Logistic regression,average_precision,0.7712,0.7676,0.0095
5,Logistic regression,accuracy,0.8532,0.8522,0.0029


In [20]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

oof_pred = cross_val_predict(
    baseline,
    X,
    y,
    cv=folds,
    method="predict",
    n_jobs=2,
)

display(pd.DataFrame(
    confusion_matrix(y, oof_pred, labels=[0, 1]),
    index=["Actual <=50K", "Actual >50K"],
    columns=["Predicted <=50K", "Predicted >50K"],
))

print(classification_report(
    y,
    oof_pred,
    target_names=["<=50K", ">50K"],
    zero_division=0,
))

,Predicted <=50K,Predicted >50K
Actual <=50K,23025,1695
Actual >50K,3118,4723


              precision    recall  f1-score   support

       <=50K       0.88      0.93      0.91     24720
        >50K       0.74      0.60      0.66      7841

    accuracy                           0.85     32561
   macro avg       0.81      0.77      0.78     32561
weighted avg       0.85      0.85      0.85     32561



In [21]:
df["workclass"].value_counts(dropna=False)

workclass
Private             22696
Self-emp-not-inc     2541
Local-gov            2093
NaN                  1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64